# # FLOAT Fast - LiveKit & Server Management
This notebook manages the setup, configuration, and execution of the FLOAT Fast backend, including LiveKit integration and public HTTPS tunneling.

## 1. Project Initialization & Setup
First, clone the project repository and check out the integration branch.

In [ ]:
# Clone project repository
!git clone https://mohamed-em2m/fast-float.git

Cloning into 'float-fast'...
remote: Enumerating objects: 3576, done.
remote: Counting objects: 100% (652/652), done.
remote: Compressing objects: 100% (433/433), done.
remote: Total 3576 (delta 358), reused 399 (delta 215), pack-reused 2924 (from 1)
Receiving objects: 100% (3576/3576), 28.63 MiB | 21.63 MiB/s, done.
Resolving deltas: 100% (2559/2559), done.


In [ ]:
%cd float-fast
!git pull

/content/float-fast
Already up to date.


## 2. Install Dependencies & Build Environments
Run the built-in setup script to configure local system dependencies, Python virtual environment, and model checkpoints.

In [ ]:
!chmod +x ./scripts/setup.sh
!./scripts/setup.sh

=== Running as root ===
=== Updating system packages ===
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
=== Installing system dependencies

## 3. Server Configuration
Configure LiveKit API settings and system credentials.

In [ ]:
%%writefile ./fast_float/config/secerts.toml
[livekit]
url = "wss://fast-float-xxxxx.livekit.cloud"                    # e.g. "wss://<your-project>.livekit.cloud" or "ws://127.0.0.1:7880"
api_key = "API"
api_secret = "KTr1t"

# --- Gemini Live voice agent (agents/) --------------------------------------
# API key for the "Gemini Live" conversation engine. The worker (agents/) and
# the API both read it from here; the environment variables GOOGLE_API_KEY /
# GEMINI_API_KEY win when set. This file is git-ignored — never commit keys.
[gemini]
api_key = "AQ."

Writing fast_float/config/secerts.toml


## 4. Run Server with Public Tunneling
Launch cloudflared for public HTTPS tunneling and start the main fast-float stack.

In [ ]:
!pip install -q uv
!uv pip install --system hypercorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
%cd float-fast
import re
import subprocess
import time

# 1. Start Cloudflare tunnel on port 3000 in background
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

time.sleep(3)
tunnel_url = None
with open("/tmp/cloudflared.log", "w") as log:
    for line in tunnel_proc.stderr:
        log.write(line)
        if "trycloudflare.com" in line:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                tunnel_url = match.group(0)
                print(f"🚀 Public API URL: {tunnel_url}")
                print(f"📖 Docs: {tunnel_url}/docs")
                break

# 2. Start FLOAT Fast server
!uv run fast_float serve --port 8000 --dual-process --frontend

Using Python 3.13.16 environment at: /usr
Checked 1 package in 99ms
(Reading database ... 128181 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.5) over (2026.10.5) ...
Setting up cloudflared (2026.10.5) ...
Processing triggers for man-db (2.12.0-4build2) ...
[Errno 2] No such file or directory: 'float-fast'
/content/float-fast
🚀 Public API URL: https://property-drove-distinguished-celebrities.trycloudflare.com
📖 Docs: https://property-drove-distinguished-celebrities.trycloudflare.com/docs
Uninstalled 1 package in 566ms
Installed 2 packages in 211ms
2026-10-28 15:04:06.067 | INFO | supervisor:10196 | fast_float.crash | crash dumps -> /content/float-fast/logs/crash-supervisor-10196.log
🚀 fast-float inference server
   API Address:      http://0.0.0.0:8000
   Dual Process:     Enabled (ZeroMQ + SHM)
   Log directory:    /content/float-fast/logs
   Studio Frontend:  http://localhost:3000
🧠 [1/3] Starting FLOAT 